# Cycle 5

Optimisation des hyperparamètres du modèle + fonction évaluation métier

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import re
import utils
import mlflow

/Users/auncly/Documents/Formation data scientist/8 - Confirmez vos compétences en MLOps/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [2]:
application = pd.read_csv('../data/rafined/application.csv', sep=',')

In [3]:
X_train, X_test, y_train, y_test = utils.split_train_data(application, 'app_TARGET')

numeric_cols = X_train.select_dtypes(include=['int64', 'int32']).columns
X_train[numeric_cols] = X_train[numeric_cols].astype('float64')
X_test[numeric_cols] = X_test[numeric_cols].astype('float64')
X_train.columns = [re.sub(r'[ \[\]\{\}:",]', '_', col) for col in X_train.columns]
X_test.columns = [re.sub(r'[ \[\]\{\}:",]', '_', col) for col in X_test.columns]

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, make_scorer

def optimize_business_cost(y_true, y_pred_proba, fn_weight=10, fp_weight=1):

    thresholds = np.linspace(0.01, 0.99, 99)
    min_cost = float('inf')

    for t in thresholds:
        y_pred = (y_pred_proba >= t).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        cost = (fn * fn_weight) + (fp * fp_weight)

        if cost < min_cost:
            min_cost = cost

    # On retourne le coût en négatif car RandomizedSearchCV cherche à maximiser
    return -min_cost

business_scorer = make_scorer(optimize_business_cost, response_method='predict_proba')

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
import mlflow
import warnings
import datetime
import scipy.stats as stats

warnings.filterwarnings("ignore")

mlflow.set_experiment("Payment_fault_classification")
mlflow.sklearn.autolog()

mlflow.end_run()

neg_count, pos_count = y_train.value_counts()
scale_weight = neg_count / pos_count
lightgbm_model = LGBMClassifier(scale_pos_weight=scale_weight)

param_distributions = {
    'model__n_estimators': stats.randint(100, 800),
    'model__learning_rate': stats.loguniform(0.01, 0.3),
    'model__num_leaves': stats.randint(20, 150),
    'model__max_depth': stats.randint(3, 12),
    'model__min_child_samples': stats.randint(10, 100),
    'model__colsample_bytree': stats.uniform(0.5, 0.5),
    'model__subsample': stats.uniform(0.5, 0.5),
    'model__reg_alpha': stats.loguniform(1e-4, 10.0),
    'model__reg_lambda': stats.loguniform(1e-4, 10.0),
}

run_name = "Optimization_LGBM_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
with mlflow.start_run(run_name=run_name) as parent_run:

    mlflow.set_tags({
        'dataset': 'rafined',
        'execution_type': 'hyperparameter_search_with_custom_scoring',
        'model_name': lightgbm_model
    })

    scoring = {
        'business_cost': business_scorer,
        'auc': 'roc_auc',
    }

    random_search = RandomizedSearchCV(
        estimator=lightgbm_model,
        param_distributions=param_distributions,
        n_iter=10,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring=scoring,
        refit='business_cost',
        n_jobs=-1,
        random_state=42
    )

    print("Début de l'optimisation des hyperparamètres...")

    random_search.fit(X_train, y_train)

    print(f"Meilleurs paramètres trouvés : {random_search.best_params_}")
    print(f"Meilleur score F1 (CV) : {random_search.best_score_}")

In [ ]:
def find_best_threshold(model, X, y_true, fn_weight=10, fp_weight=1):
    y_pred_proba = model.predict_proba(X)[:, 1]
    thresholds = np.linspace(0.01, 0.99, 99)

    min_cost = float('inf')
    best_t = 0.5

    for t in thresholds:
        print(f"Seuil : {t} : ")
        y_pred = (y_pred_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        cost = (fn * fn_weight) + (fp * fp_weight)
        print(confusion_matrix(y_true, y_pred))
        if cost < min_cost:
            min_cost = cost
            best_t = t

    return best_t, min_cost

In [ ]:
best_model = random_search.best_estimator_

optimal_threshold, final_cost = find_best_threshold(best_model, X_train, y_train)

print(f"Seuil de décision optimal à utiliser en production : {optimal_threshold:.2f}")

# On logge manuellement cette information métier cruciale dans MLflow
with mlflow.start_run(run_id=mlflow.last_active_run().info.run_id):
    mlflow.log_param("optimal_decision_threshold", optimal_threshold)
    mlflow.log_metric("final_train_business_cost", final_cost)


In [ ]:
best_model = random_search.best_estimator_
importances = best_model.feature_importances_
features_names = best_model.feature_name_

utils.show_importances(importances, features_names, 20)

print(features_names[:20])

In [ ]:
df_importance = pd.DataFrame({
    'Variable': features_names,
    'Importance': importances
})

df_importance = df_importance.sort_values(by='Importance', ascending=False)

df_importance['Variable'].head(20).tolist()

In [ ]:
local_path = mlflow.artifacts.download_artifacts(
    artifact_uri="models:/payment_fault_classificator/1",
    dst_path="./mon_modele_local"
)

print(f"Modèle téléchargé ici : {local_path}")

In [10]:
sample = X_test.sample(1)
sample.to_json()

'{"SK_ID_CURR":{"49584":157400.0},"app_CNT_CHILDREN":{"49584":2.0},"app_AMT_INCOME_TOTAL":{"49584":135000.0},"app_AMT_CREDIT":{"49584":677664.0},"app_AMT_ANNUITY":{"49584":37971.0},"app_AMT_GOODS_PRICE":{"49584":585000.0},"app_REGION_POPULATION_RELATIVE":{"49584":0.019101},"app_DAYS_BIRTH":{"49584":-11744.0},"app_DAYS_EMPLOYED":{"49584":-138.0},"app_DAYS_REGISTRATION":{"49584":-281.0},"app_DAYS_ID_PUBLISH":{"49584":-2223.0},"app_OWN_CAR_AGE":{"49584":10.0},"app_FLAG_MOBIL":{"49584":1.0},"app_FLAG_EMP_PHONE":{"49584":1.0},"app_FLAG_WORK_PHONE":{"49584":0.0},"app_FLAG_CONT_MOBILE":{"49584":1.0},"app_FLAG_PHONE":{"49584":0.0},"app_FLAG_EMAIL":{"49584":1.0},"app_CNT_FAM_MEMBERS":{"49584":4.0},"app_REGION_RATING_CLIENT":{"49584":2.0},"app_REGION_RATING_CLIENT_W_CITY":{"49584":2.0},"app_HOUR_APPR_PROCESS_START":{"49584":12.0},"app_REG_REGION_NOT_LIVE_REGION":{"49584":0.0},"app_REG_REGION_NOT_WORK_REGION":{"49584":0.0},"app_LIVE_REGION_NOT_WORK_REGION":{"49584":0.0},"app_REG_CITY_NOT_LIVE_CIT